# Binary Quadratic Model formulation

A **Binary Quadratic Model (BQM)** is the native language of quantum
annealing. It encodes an objective function over binary variables as
linear biases and pairwise interactions:

$$
E(\mathbf{x}) = \sum_i h_i x_i + \sum_{i<j} J_{ij} x_i x_j + c
$$

for SPIN variables ($x_i \in \{-1,+1\}$) or BINARY variables
($x_i \in \{0,1\}$). This notebook builds a small BQM, inspects it,
and visualises the interaction graph with networkx.

This notebook is self-contained. It does not import `bqm_formulation.py`.

In [ ]:
import dimod
import matplotlib.pyplot as plt
import networkx as nx

## Build the BQM

We use SPIN variables and four variables `{a, b, c, d}`. Negative
linear biases favour $+1$, positive biases favour $-1$. Negative
quadratic couplings favour aligned spins, positive couplings favour
anti-aligned spins.

In [ ]:
bqm = dimod.BinaryQuadraticModel(
    {"a": -2.0, "b": -1.5, "c": 1.0, "d": 0.5},
    {("a", "b"): -1.0, ("b", "c"): 0.8, ("a", "c"): 0.5, ("c", "d"): -0.3},
    0.0,
    "SPIN",
)

print(f"Variables: {list(bqm.variables)}")
print(f"Linear terms:    {dict(bqm.linear)}")
print(f"Quadratic terms: {dict(bqm.quadratic)}")
print(f"Offset:          {bqm.offset}")
print(f"vartype:         {bqm.vartype}")

## Evaluate sample energies

The `energy` method computes $E(\mathbf{x})$ for any spin configuration.

In [ ]:
samples = [
    {"a": 1, "b": 1, "c": -1, "d": 1},
    {"a": -1, "b": -1, "c": 1, "d": -1},
    {"a": 1, "b": -1, "c": 1, "d": 1},
]
print("Sample                          Energy")
for s in samples:
    print(f"  {s}  ->  E = {bqm.energy(s):+.2f}")

## Visualise the interaction graph

Node colour encodes the sign of the linear bias; edge colour encodes
the sign of the coupling (green = ferromagnetic, red = anti-ferromagnetic).

In [ ]:
G = nx.Graph()
for v in bqm.variables:
    G.add_node(v, bias=float(bqm.linear[v]))
for (u, v), weight in bqm.quadratic.items():
    G.add_edge(u, v, weight=weight)

pos = nx.spring_layout(G, seed=42)
biases = [G.nodes[n]["bias"] for n in G.nodes]
edge_weights = [G.edges[e]["weight"] for e in G.edges]
edge_colors = ["green" if w < 0 else "red" for w in edge_weights]
node_sizes = [800 + 400 * abs(b) for b in biases]
node_colors = ["lightblue" if b < 0 else "salmon" for b in biases]

fig, ax = plt.subplots(figsize=(7, 5))
nx.draw_networkx_nodes(G, pos, ax=ax, node_color=node_colors, node_size=node_sizes)
nx.draw_networkx_labels(G, pos, ax=ax, font_size=12, font_weight="bold")
nx.draw_networkx_edges(G, pos, ax=ax, width=2, edge_color=edge_colors)
edge_labels = {(u, v): f"{w:+.1f}" for (u, v), w in G.edges.items()}
nx.draw_networkx_edge_labels(G, pos, ax=ax, edge_labels=edge_labels, font_size=10)
ax.set_title("BQM interaction graph\n(blue = negative bias, red = positive bias)")
ax.axis("off")
plt.tight_layout()
plt.show()